In [0]:

%pip install faker scikit-learn xgboost lifelines plotly seaborn mlflow

In [0]:
dbutils.library.restartPython()


In [0]:
# ── Database setup (hive_metastore — works on Community Edition) ─────────────
DB = "hospital_blackhole"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB}")
spark.sql(f"USE {DB}")
print(f"Database set to: {DB}")
 
# ── Paths (use DBFS which is available on Community Edition) ─────────────────
BASE_PATH = f"dbfs:/user/hive/warehouse/{DB}"
print(f"Data path: {BASE_PATH}")


In [0]:
# ── Shared config dict ───────────────────────────────────────────────────────
config = {
    "db": DB,
 
    # Bronze tables
    "bronze_hmis_referrals"   : f"{DB}.bronze_hmis_referrals",
    "bronze_facility_master"  : f"{DB}.bronze_facility_master",
    "bronze_pmjay_claims"     : f"{DB}.bronze_pmjay_claims",
    "bronze_nfhs_baseline"    : f"{DB}.bronze_nfhs_baseline",
    "bronze_disease_burden"   : f"{DB}.bronze_disease_burden",
 
    # Silver tables
    "silver_patient_journeys" : f"{DB}.silver_patient_journeys",
    "silver_dropout_events"   : f"{DB}.silver_dropout_events",
    "silver_facility_capacity": f"{DB}.silver_facility_capacity",
    "silver_geo_barriers"     : f"{DB}.silver_geo_barriers",
    "silver_survival_features": f"{DB}.silver_survival_features",
 
    # Gold tables
    "gold_dropout_funnel"     : f"{DB}.gold_dropout_funnel",
    "gold_bhs_district"       : f"{DB}.gold_bhs_district",
    "gold_stage_heatmap"      : f"{DB}.gold_stage_heatmap",
    "gold_survival_curves"    : f"{DB}.gold_survival_curves",
    "gold_facility_gap"       : f"{DB}.gold_facility_gap",
    "gold_forecast"           : f"{DB}.gold_forecast",
    "gold_india_summary"      : f"{DB}.gold_india_summary",
 
    # Settings
    "n_districts"      : 50,    # reduced for Community Edition performance
    "shuffle_partitions": 8,    # single-node friendly
    "seed"             : 42,
}

In [0]:
# ── Spark tuning for single-node Community Edition ───────────────────────────
spark.conf.set("spark.sql.shuffle.partitions",
               str(config["shuffle_partitions"]))
# spark.databricks.delta.optimizeWrite.enabled  ← NOT available on Community Edition, removed
print("Spark config applied for single-node Community Edition.")

In [0]:
import mlflow
mlflow.set_experiment("/Users/diyasaxena26@gmail.com/hospital_blackhole_survival_model")
print("MLflow experiment set.")

In [0]:
print("="*50)
print("CLUSTER INFO")
print("="*50)

# Get Spark configs safely
sc_conf = spark.conf.getAll   
for key, val in sc_conf.items():
    if "cores" in key.lower() or "memory" in key.lower() or "executor" in key.lower():
        print(f"{key}: {val}")

print(f"\nSpark version : {spark.version}")
print(f"Active DB     : {DB}")
print(f"Partitions    : {config['shuffle_partitions']}")

print("="*50)
print("✅ Setup complete. Run notebooks in order: 0 → 1 → 2 → 3 → 4 → 5")